# **Velocity, Recency, Frequency, and Monetary (VRFM) Analysis**

**VRFM** is a set of features often used in **customer behavior analysis, fraud detection, and risk assessment**. Below is a theoretical explanation of each term and how to compute it using a **transaction dataset** in a Jupyter Notebook.

---

## **1. Understanding VRFM Metrics**

### **(a) Velocity (Transaction Velocity)**
- Measures **how fast transactions are happening** over time.  
- **Formula:**  
  $$ 
  \text{Velocity} = \frac{\text{Number of transactions in a given time window}}{\text{Time window (hours/days)}}
  $$
- **Use Case:**  
  - High velocity may indicate **fraud** (e.g., multiple transactions in a short period).  

---

### **(b) Recency**
- Measures **how recently the last transaction occurred**.  
- **Formula:**  
  $$
  \text{Recency} = \text{Current Date} - \text{Last Transaction Date}
  $$
- **Use Case:**  
  - In **fraud detection**, a sudden drop in recency (recent unusual transactions) may indicate risk.  
  - In **customer segmentation**, low recency means **active customers**, while high recency means **dormant customers**.  

---

### **(c) Frequency**
- Measures **how often transactions occur** in a given period.  
- **Formula:**  
  $$
  \text{Frequency} = \text{Total Transactions per User}
  $$
- **Use Case:**  
  - Frequent transactions in a short period might be **suspicious**.  
  - Can be used for **customer segmentation** (high-frequency customers vs. inactive ones).  

---

### **(d) Monetary (Transaction Amount)**
- Measures **the total amount spent by a user** over a given period.  
- **Formula:**  
  $$
  \text{Monetary} = \sum (\text{Transaction Amount})
  $$
- **Use Case:**  
  - Helps detect **high-value transactions** that deviate from the normal spending pattern.  
  - Used in **customer value assessment** (High-value customers vs. Low-value customers).

### **2. Implementing VRFM in Jupyter Notebook**

In [97]:
import pandas as pd

In [98]:
df = pd.read_csv('/Users/shivendragupta/Desktop/Machine Learning/bank_card_transactions-1.csv')

In [99]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount
0,4000498670561373,2023-09-20 05:26:00,54616,52.03
1,4000498670561373,2023-04-23 10:42:00,17076,414.40
2,4000498670561373,2023-06-02 03:33:00,46552,1106.05
3,4000498670561373,2023-08-04 19:37:00,72977,4789.17
4,4000498670561373,2023-10-20 08:08:00,32949,2782.97


### Converted timestamp from obj to Datetime

In [100]:
df['datetime'] = pd.to_datetime(df['Timestamp'])

### Sorted Values based on Card Number and datetime

In [101]:
df = df.sort_values(by=["Card_Number", "datetime"])

### Added floor with respect to hour of each datetime and added a new column

In [102]:
df["hour"] = df["datetime"].dt.floor("h")

# Velocity

### Added a velocity column .
* Velocity - Velocity is defined as number of transaction per hour per day

In [103]:
velocity = df.groupby(["Card_Number", "hour"]).size().reset_index(name="velocity")

### Merged the velocity and df 

In [104]:
df = df.merge(velocity, on=["Card_Number", "hour"], how="left")

In [105]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1


# Recency

### Added a Recency column
* Recency - Time between transactions. This can be calculated either in minutes, hrs , days or months.

In [106]:
df["prev_transaction_time"] = df.groupby("Card_Number")["datetime"].shift(1)

In [107]:
df["recency"] = (df["datetime"] - df["prev_transaction_time"]).dt.total_seconds() / (3600*24)

In [108]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity,prev_transaction_time,recency
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1,NaT,NaN
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1,2023-04-29 02:28:00,28.131250
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1,2023-05-27 05:37:00,21.472917
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1,2023-06-17 16:58:00,62.153472
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1,2023-08-18 20:39:00,10.532639


### Added the frequency column 
* Frequency - This can be defined as number of transaction per day , week or month depending upon the need.

In [109]:
df['date'] = df['datetime'].dt.date

In [110]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity,prev_transaction_time,recency,date
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1,NaT,NaN,2023-04-29
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1,2023-04-29 02:28:00,28.131250,2023-05-27
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1,2023-05-27 05:37:00,21.472917,2023-06-17
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1,2023-06-17 16:58:00,62.153472,2023-08-18
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1,2023-08-18 20:39:00,10.532639,2023-08-29


In [111]:
frequency = df.groupby(["Card_Number", "date"]).size().reset_index(name="frequency")

In [112]:
df = df.merge(frequency, on=["Card_Number", "date"], how="left")

In [113]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity,prev_transaction_time,recency,date,frequency
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1,NaT,NaN,2023-04-29,1
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1,2023-04-29 02:28:00,28.131250,2023-05-27,1
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1,2023-05-27 05:37:00,21.472917,2023-06-17,1
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1,2023-06-17 16:58:00,62.153472,2023-08-18,1
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1,2023-08-18 20:39:00,10.532639,2023-08-29,1


# Monetary

### Added a monetary column 
* Monetary - This can be defined as sum of transactions per card per date

In [114]:
monetary = df.groupby(["Card_Number", "date"])["Amount"].sum().reset_index(name="monetary")

df = df.merge(monetary, on=["Card_Number", "date"], how="left")


In [115]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity,prev_transaction_time,recency,date,frequency,monetary
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1,NaT,NaN,2023-04-29,1,2847.42
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1,2023-04-29 02:28:00,28.131250,2023-05-27,1,4588.56
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1,2023-05-27 05:37:00,21.472917,2023-06-17,1,3045.13
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1,2023-06-17 16:58:00,62.153472,2023-08-18,1,4521.77
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1,2023-08-18 20:39:00,10.532639,2023-08-29,1,4738.93


# Standard Deviation

### Added a std_deviation Column
 * std - tells that any transaction with higher starndard deviation can be a potential fraud

In [116]:
std_amount = df.groupby("Card_Number")["Amount"].transform("std")
df["std_transaction_amount"] = std_amount


In [117]:
df.head()

,Card_Number,Timestamp,Zip_Code,Amount,datetime,hour,velocity,prev_transaction_time,recency,date,frequency,monetary,std_transaction_amount
0,4000407982502282,2023-04-29 02:28:00,72977,2847.42,2023-04-29 02:28:00,2023-04-29 02:00:00,1,NaT,NaN,2023-04-29,1,2847.42,1218.570293
1,4000407982502282,2023-05-27 05:37:00,87703,4588.56,2023-05-27 05:37:00,2023-05-27 05:00:00,1,2023-04-29 02:28:00,28.131250,2023-05-27,1,4588.56,1218.570293
2,4000407982502282,2023-06-17 16:58:00,93858,3045.13,2023-06-17 16:58:00,2023-06-17 16:00:00,1,2023-05-27 05:37:00,21.472917,2023-06-17,1,3045.13,1218.570293
3,4000407982502282,2023-08-18 20:39:00,71553,4521.77,2023-08-18 20:39:00,2023-08-18 20:00:00,1,2023-06-17 16:58:00,62.153472,2023-08-18,1,4521.77,1218.570293
4,4000407982502282,2023-08-29 09:26:00,71553,4738.93,2023-08-29 09:26:00,2023-08-29 09:00:00,1,2023-08-18 20:39:00,10.532639,2023-08-29,1,4738.93,1218.570293
